In [19]:
from pathlib import Path
import pandas as pd

data_path = Path("Anuran Calls (MFCCs)") / "Frogs_MFCCs.csv"
labels = ["Family", "Genus", "Species"]
dropped_cols = ["RecordID"]
test_size = 0.3

frogs = pd.read_csv(data_path).drop(columns=dropped_cols)
X, Y = frogs.drop(columns=labels), frogs[labels]
print(X.shape, Y.shape)

from sklearn.model_selection import train_test_split

X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=test_size, random_state=42)
print(X_train.shape, X_test.shape)

(7195, 22) (7195, 3)
(5036, 22) (2159, 22)


**Question 1b**

In [20]:
import numpy as np

def exact_match(Y_true, Y_pred):
    return np.mean(np.all(np.asarray(Y_true) == np.asarray(Y_pred), axis=1))

def hamming_loss(Y_true, Y_pred):
    return np.mean(np.asarray(Y_true) != np.asarray(Y_pred))

In [21]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.multiclass import OneVsRestClassifier
from sklearn.svm import SVC

def make_rbf_svm(scale):
    return Pipeline([("scale", StandardScaler() if scale else "passthrough"),
                     ("clf", OneVsRestClassifier(SVC(kernel="rbf")))])

quick = False
cv_folds = 10
c_grid = np.logspace(-1, 4, 3 if quick else 6)
sigma_grid = np.linspace(0.1, 2, 3 if quick else 10)
param_grid = {"clf__estimator__C": c_grid,
              "clf__estimator__gamma": 1 / (2 * sigma_grid**2)}


In [22]:
for C in [c_grid.min(), c_grid.max()]:
    for sigma in [sigma_grid.min(), sigma_grid.max()]:
        model = make_rbf_svm(scale=False).set_params(clf__estimator__C=C,
                                                     clf__estimator__gamma=1 / (2 * sigma**2))
        train_acc = model.fit(X_train, Y_train["Species"]).score(X_train, Y_train["Species"])
        print(f"C={C:<8g} sigma={sigma:.2f}  train_acc={train_acc:.3f}")

C=0.1      sigma=0.10  train_acc=0.836
C=0.1      sigma=2.00  train_acc=0.870
C=10000    sigma=0.10  train_acc=1.000
C=10000    sigma=2.00  train_acc=1.000


In [23]:
from sklearn.model_selection import GridSearchCV

def fit_binary_relevance(make_model, param_grid, scale):
    Y_pred = pd.DataFrame(index=Y_test.index)
    for label in labels:
        search = GridSearchCV(make_model(scale), param_grid, cv=cv_folds, n_jobs=-1)
        search.fit(X_train, Y_train[label])
        Y_pred[label] = search.predict(X_test)
        print(f"{label:<8} best={search.best_params_}  cv_acc={search.best_score_:.3f}")
    return Y_pred

for scale in [False, True]:
    Y_pred_rbf = fit_binary_relevance(make_rbf_svm, param_grid, scale)
    print(f"scaled={scale}  exact_match={exact_match(Y_test, Y_pred_rbf):.3f}  "
          f"hamming_loss={hamming_loss(Y_test, Y_pred_rbf):.3f}\n")

Family   best={'clf__estimator__C': np.float64(10.0), 'clf__estimator__gamma': np.float64(1.8334087822544134)}  cv_acc=0.993
Genus    best={'clf__estimator__C': np.float64(10.0), 'clf__estimator__gamma': np.float64(1.8334087822544134)}  cv_acc=0.991
Species  best={'clf__estimator__C': np.float64(10.0), 'clf__estimator__gamma': np.float64(1.8334087822544134)}  cv_acc=0.990
scaled=False  exact_match=0.989  hamming_loss=0.008

Family   best={'clf__estimator__C': np.float64(10.0), 'clf__estimator__gamma': np.float64(0.125)}  cv_acc=0.990
Genus    best={'clf__estimator__C': np.float64(100.0), 'clf__estimator__gamma': np.float64(0.125)}  cv_acc=0.987
Species  best={'clf__estimator__C': np.float64(10.0), 'clf__estimator__gamma': np.float64(0.125)}  cv_acc=0.986
scaled=True  exact_match=0.983  hamming_loss=0.012



Gaussian SVM. 10-fold CV per label over C ∈ {0.1, 31.6, 10⁴} and σ ∈ {0.1, 1.05, 2} (γ = 1/2σ²).

**Question 1b.iii**

In [24]:
L1_C_GRID = np.logspace(-3, 3, 13)    
L1_PARAM_GRID = {"clf__C": L1_C_GRID}

In [25]:
from sklearn.svm import LinearSVC

def make_l1_svm(scale=True):
    return Pipeline([("scale", StandardScaler()),
                     ("clf", LinearSVC(penalty="l1", dual=False, max_iter=10000))])

In [26]:
Y_pred_l1 = fit_binary_relevance(make_l1_svm, L1_PARAM_GRID, scale=True)
print(f"exact_match={exact_match(Y_test, Y_pred_l1):.3f}  "
      f"hamming_loss={hamming_loss(Y_test, Y_pred_l1):.3f}")

Family   best={'clf__C': np.float64(1.0)}  cv_acc=0.941
Genus    best={'clf__C': np.float64(10.0)}  cv_acc=0.953
Species  best={'clf__C': np.float64(10.0)}  cv_acc=0.960
exact_match=0.913  hamming_loss=0.057


**Question 1b.iv**

In [32]:
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE
from sklearn.metrics import balanced_accuracy_score


def make_smote_l1_svm(scale=True):
    return ImbPipeline([("scale", StandardScaler()),
                        ("smote", SMOTE(random_state=42)),
                        ("clf", LinearSVC(penalty="l1", dual=False, max_iter=10000))])

In [34]:
Y_pred_smote = fit_binary_relevance(make_smote_l1_svm, L1_PARAM_GRID, scale=True)
print(f"exact_match={exact_match(Y_test, Y_pred_smote):.3f}  "
      f"hamming_loss={hamming_loss(Y_test, Y_pred_smote):.3f}")


for label in labels:
    print(f"{label:<8} L1={balanced_accuracy_score(Y_test[label], Y_pred_l1[label]):.3f}  "
          f"L1+SMOTE={balanced_accuracy_score(Y_test[label], Y_pred_smote[label]):.3f}")

Family   best={'clf__C': np.float64(3.1622776601683795)}  cv_acc=0.921


KeyboardInterrupt: 